In [1]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path(r"E:\Littering_event_detection")

print("Project folder exists:", PROJECT_ROOT.exists())
print("Event file exists:", (
    PROJECT_ROOT / "src" / "event" / "event_state_machine.py"
).exists())

if PROJECT_ROOT.exists():
    sys.path.insert(0, str(PROJECT_ROOT))

    print("Project folders:", [
        p.name for p in PROJECT_ROOT.iterdir() if p.is_dir()
    ])

Project folder exists: True
Event file exists: True
Project folders: ['.git', '.venv', 'backend', 'camera', 'configs', 'data', 'database', 'incidents', 'models', 'notebooks', 'outputs', 'screens', 'src', 'tests', 'venv']


In [2]:
from src.event.event_state_machine import (
    EventObservation,
    EventState,
    EventStateMachine,
)

print("Event state machine imported successfully!")
print("Available states:", [state.value for state in EventState])

Event state machine imported successfully!
Available states: ['NO_EVENT', 'HOLDING_WASTE', 'RELEASE_CANDIDATE', 'WASTE_LANDED', 'PERSON_LEFT', 'LITTERING_CANDIDATE', 'PROPER_DISPOSAL', 'PICKUP', 'NO_LITTERING', 'UNCERTAIN']


In [3]:
machine = EventStateMachine()

print("State machine created successfully!")
print("Initial state:", machine.state)

State machine created successfully!
Initial state: EventState.NO_EVENT


In [4]:
from datetime import datetime, timedelta, timezone
from src.event.event_state_machine import (
    EventObservation,
    EventState,
    EventStateMachine,
)

machine = EventStateMachine(min_event_confidence=0.65)
start = datetime.now(timezone.utc)

observations = [
    {
        "label": "Person holding waste",
        "holding_observed": True,
        "release_observed": False,
        "waste_landed_observed": False,
        "person_left_observed": False,
    },
    {
        "label": "Waste released",
        "holding_observed": False,
        "release_observed": True,
        "waste_landed_observed": False,
        "person_left_observed": False,
    },
    {
        "label": "Waste landed",
        "holding_observed": False,
        "release_observed": False,
        "waste_landed_observed": True,
        "person_left_observed": False,
    },
    {
        "label": "Person left",
        "holding_observed": False,
        "release_observed": False,
        "waste_landed_observed": False,
        "person_left_observed": True,
    },
]

for i, item in enumerate(observations):
    obs = EventObservation(
        timestamp=start + timedelta(seconds=i + 1),
        person_track_id=1,
        waste_track_id=2,
        holding_observed=item["holding_observed"],
        release_observed=item["release_observed"],
        waste_landed_observed=item["waste_landed_observed"],
        person_left_observed=item["person_left_observed"],
        observation_confidence=0.90,
    )

    result = machine.update(obs)
    print(f'{item["label"]}: {result}')

Person holding waste: None
Waste released: None
Waste landed: None
Person left: EventRecord(person_track_id=1, waste_track_id=2, event_type='LITTERING_CANDIDATE', start_timestamp=datetime.datetime(2026, 10, 10, 17, 5, 45, 618189, tzinfo=datetime.timezone.utc), end_timestamp=datetime.datetime(2026, 10, 10, 17, 5, 48, 618189, tzinfo=datetime.timezone.utc), event_confidence=0.9, evidence_timeline=['waste held', 'waste released', 'waste landed', 'person left', 'littering sequence supported'], review_required=True, state='LITTERING_CANDIDATE')


In [5]:
from datetime import datetime, timedelta, timezone
from src.event.event_state_machine import EventObservation, EventStateMachine

def run_scenario(label, **signals):
    machine = EventStateMachine(min_event_confidence=0.65)
    now = datetime.now(timezone.utc)

    observations = [
        EventObservation(
            timestamp=now + timedelta(seconds=i + 1),
            person_track_id=1,
            waste_track_id=2,
            observation_confidence=0.90,
            **signal,
        )
        for i, signal in enumerate(signals["observations"])
    ]

    result = None
    for obs in observations:
        result = machine.update(obs)

    print(f"{label}: {result}")

run_scenario(
    "Proper disposal",
    observations=[
        {"entered_bin_observed": True},
    ],
)

run_scenario(
    "Waste pickup",
    observations=[
        {"pickup_observed": True},
    ],
)

Proper disposal: None
Waste pickup: None


In [6]:
from datetime import datetime, timedelta, timezone
from src.event.event_state_machine import EventObservation, EventStateMachine

def test_outcome(outcome_name, final_signal):
    machine = EventStateMachine(min_event_confidence=0.65)
    start = datetime.now(timezone.utc)

    signals = [
        {"holding_observed": True},
        {"release_observed": True},
        {"waste_landed_observed": True},
        final_signal,
    ]

    result = None

    for i, signal in enumerate(signals):
        obs = EventObservation(
            timestamp=start + timedelta(seconds=i + 1),
            person_track_id=1,
            waste_track_id=2,
            observation_confidence=0.90,
            **signal,
        )
        result = machine.update(obs)
        print(f"  Step {i + 1}: {result}")

    print(f"{outcome_name} final result: {result}\n")

test_outcome(
    "Proper disposal",
    {"entered_bin_observed": True},
)

test_outcome(
    "Waste pickup",
    {"pickup_observed": True},
)

  Step 1: None
  Step 2: None
  Step 3: None
  Step 4: EventRecord(person_track_id=1, waste_track_id=2, event_type='PROPER_DISPOSAL', start_timestamp=datetime.datetime(2026, 10, 10, 17, 7, 3, 109264, tzinfo=datetime.timezone.utc), end_timestamp=datetime.datetime(2026, 10, 10, 17, 7, 6, 109264, tzinfo=datetime.timezone.utc), event_confidence=0.9, evidence_timeline=['waste held', 'waste released', 'waste landed', 'waste entered bin'], review_required=False, state='PROPER_DISPOSAL')
Proper disposal final result: EventRecord(person_track_id=1, waste_track_id=2, event_type='PROPER_DISPOSAL', start_timestamp=datetime.datetime(2026, 10, 10, 17, 7, 3, 109264, tzinfo=datetime.timezone.utc), end_timestamp=datetime.datetime(2026, 10, 10, 17, 7, 6, 109264, tzinfo=datetime.timezone.utc), event_confidence=0.9, evidence_timeline=['waste held', 'waste released', 'waste landed', 'waste entered bin'], review_required=False, state='PROPER_DISPOSAL')

  Step 1: None
  Step 2: None
  Step 3: None
  Step 4:

In [7]:
from datetime import datetime, timedelta, timezone
from src.event.event_state_machine import EventObservation, EventStateMachine

machine = EventStateMachine(min_event_confidence=0.65)
start = datetime.now(timezone.utc)

observations = [
    EventObservation(
        timestamp=start + timedelta(seconds=1),
        person_track_id=1,
        waste_track_id=2,
        holding_observed=True,
        observation_confidence=0.90,
    ),
    EventObservation(
        timestamp=start + timedelta(seconds=2),
        person_track_id=1,
        waste_track_id=2,
        release_observed=True,
        observation_confidence=0.90,
    ),
    EventObservation(
        timestamp=start + timedelta(seconds=3),
        person_track_id=1,
        waste_track_id=2,
        waste_landed_observed=True,
        evidence_incomplete=True,
        observation_confidence=0.50,
    ),
]

for obs in observations:
    result = machine.update(obs)
    print(result)

None
None
EventRecord(person_track_id=1, waste_track_id=2, event_type='UNCERTAIN', start_timestamp=datetime.datetime(2026, 10, 10, 17, 9, 18, 722235, tzinfo=datetime.timezone.utc), end_timestamp=datetime.datetime(2026, 10, 10, 17, 9, 20, 722235, tzinfo=datetime.timezone.utc), event_confidence=0.767, evidence_timeline=['waste held', 'waste released', 'evidence incomplete or occluded'], review_required=True, state='UNCERTAIN')


In [8]:
from datetime import datetime, timedelta, timezone
from src.event.event_state_machine import EventObservation, EventStateMachine

machine = EventStateMachine(min_event_confidence=0.65)
start = datetime.now(timezone.utc)

steps = [
    {"holding_observed": True},
    {"release_observed": True},
    {"waste_landed_observed": True},
    {"person_left_observed": True},
]

result = None

for i, signals in enumerate(steps):
    obs = EventObservation(
        timestamp=start + timedelta(seconds=i + 1),
        person_track_id=1,
        waste_track_id=2,
        observation_confidence=0.40,
        **signals,
    )
    result = machine.update(obs)
    print(f"Step {i + 1}: {result}")

print("\nFinal event type:", result.event_type if result else None)
print("Final confidence:", result.event_confidence if result else None)
print("Review required:", result.review_required if result else None)

Step 1: None
Step 2: None
Step 3: None
Step 4: EventRecord(person_track_id=1, waste_track_id=2, event_type='UNCERTAIN', start_timestamp=datetime.datetime(2026, 10, 10, 17, 9, 49, 926392, tzinfo=datetime.timezone.utc), end_timestamp=datetime.datetime(2026, 10, 10, 17, 9, 52, 926392, tzinfo=datetime.timezone.utc), event_confidence=0.4, evidence_timeline=['waste held', 'waste released', 'waste landed', 'person left'], review_required=True, state='UNCERTAIN')

Final event type: UNCERTAIN
Final confidence: 0.4
Review required: True


In [9]:
from datetime import datetime, timedelta, timezone
from src.event.event_state_machine import EventObservation, EventStateMachine

machine = EventStateMachine(min_event_confidence=0.65)
start = datetime.now(timezone.utc)

steps = [
    {"person_track_id": 1, "holding_observed": True},
    {"person_track_id": 1, "release_observed": True},
    {"person_track_id": 2, "waste_landed_observed": True},
]

result = None

for i, step in enumerate(steps):
    person_id = step.pop("person_track_id")

    obs = EventObservation(
        timestamp=start + timedelta(seconds=i + 1),
        person_track_id=person_id,
        waste_track_id=2,
        observation_confidence=0.90,
        **step,
    )

    result = machine.update(obs)
    print(f"Step {i + 1}: {result}")

print("\nFinal result:", result)

Step 1: None
Step 2: None
Step 3: EventRecord(person_track_id=1, waste_track_id=2, event_type='UNCERTAIN', start_timestamp=datetime.datetime(2026, 10, 10, 17, 10, 20, 64303, tzinfo=datetime.timezone.utc), end_timestamp=datetime.datetime(2026, 10, 10, 17, 10, 22, 64303, tzinfo=datetime.timezone.utc), event_confidence=0.9, evidence_timeline=['waste held', 'waste released'], review_required=True, state='UNCERTAIN')

Final result: EventRecord(person_track_id=1, waste_track_id=2, event_type='UNCERTAIN', start_timestamp=datetime.datetime(2026, 10, 10, 17, 10, 20, 64303, tzinfo=datetime.timezone.utc), end_timestamp=datetime.datetime(2026, 10, 10, 17, 10, 22, 64303, tzinfo=datetime.timezone.utc), event_confidence=0.9, evidence_timeline=['waste held', 'waste released'], review_required=True, state='UNCERTAIN')
